# 既存データセットの活用
---
## 目的
Pytorchのモジュールtorchvisionにおける既存データセットの活用方法．

## torchvisionにおける既存データセット
torchvisionは，PyTorchのコンピュータビジョンライブラリである．<br>
torchvisionでは，画像分類タスクの代表的な次のデータセットが利用できる．
- MNIST
- CIFAR10
- CIFAR-100
- ImageNet

## データセットの利用
### 前準備
はじめに，必要なパッケージのインポートとGPUの使用確認を行う．<br>
GPUが使用できるかどうかは，`torch.cuda.is_available()`で確認できる．

In [2]:
import os
import numpy as np
import torch
import torch.nn as nn

import torchvision
import torchvision.transforms as transforms

# GPUを利用しているか確認
iscuda = torch.cuda.is_available()
print('CUDA is ', iscuda)

CUDA is  True


### データセットのダウンロード
ここでは簡単な例として，MNISTデータセットを利用する．<br>
1. MNISTデータセットの呼び出し．<br>
MNISTデータセットは`torchvision.datasets.MNIST`クラスを用いることで簡単に利用することができる．<br>
`torchvision.datasets.MNIST`には以下の引数がある．<br>
- `root`: MNISTデータセットがある（or ダウンロードしたい）ディレクトリまでのパス
- `train`: True -> 学習用データ，False ->テストデータ
- `download`: rootで指定したパスにMNISTデータセットがない場合ダウンロードするかどうか
- `transform`: データに対して適用する前処理を定義
<br>

`transform`には次のクラス（一例を示す）がある．
- `ToTensor()`: 呼び出したデータをTensor型に変換するクラス
- `Grayscale()`: 画像をグレースケール変換するクラス
- `Normalize()`: データを正規化するクラス
- `RandomCrop()`: データをランダムクロップするクラス

In [3]:
# 訓練データ
train_data = torchvision.datasets.MNIST(root="./dataset", train=True, transform=transforms.ToTensor(), download=True)
# テストデータ
test_data = torchvision.datasets.MNIST(root="./dataset", train=False, transform=transforms.ToTensor(), download=True)

# データタイプの表示
print('訓練データ/ラベルのタイプ: ',type(train_data.data), type(train_data.targets))
print('訓練データ/ラベルのサイズ: ',train_data.data.size(), train_data.targets.size())
print('テストデータ/ラベルのタイプ: ',type(test_data.data), type(test_data.targets))
print('テストデータ/ラベルのサイズ: ',test_data.data.size(), test_data.targets.size())

<class 'torch.Tensor'> <class 'torch.Tensor'>
<class 'torch.Tensor'> <class 'torch.Tensor'>
torch.Size([60000, 28, 28]) torch.Size([60000])
torch.Size([10000, 28, 28]) torch.Size([10000])


2. データセットから$i$番目のデータの呼び出し

In [4]:
# 訓練データセットのデータ番号0の呼び出し
img_data, label_data = train_data[0]
print(type(img_data), img_data.size())
print(type(label_data), label_data)

The number of training data: 60000
The number of test data: 10000
<class 'torch.Tensor'> torch.Size([1, 28, 28])
<class 'int'> 5


$0$番目のデータ`train_data[0]`は，配列サイズが`[1,28,28]`である．<br>
前処理で指定した`ToTensor()`は，Tensor型にデータを変換する際に配列形式を[チャンネル, 縦, 横]の順番に配列操作を行う．<br>

3. DataLoaderを用いたデータの呼び出し<br>
Pytorchでは，`torchvision.datasets.MNIST`クラスで呼び出したデータによる効率的な活用をするために`torch.utils.data.DataLoader`が用意されている．
`torch.utils.data.DataLoader`には以下の引数（一部を示す）がある．<br>
- `dataset`: データセットクラスのインスタンスを指定
- `batch_size`: ミニバッチサイズを指定
- `shuffle`: データをランダムにサンプルするかどうか

In [5]:
# DataLoaderの定義
train_loader = torch.utils.data.DataLoader(dataset=train_data, batch_size=10, shuffle=True)
test_loader = torch.utils.data.DataLoader(dataset=test_data, batch_size=10, shuffle=False)

for img, label in train_loader:
    print('訓練データ/ラベルのタイプ: ',img.size(),label.size())
    print('サンプルされたミニバッチ数のラベル: ', label)
    break

torch.Size([10, 1, 28, 28])
torch.Size([10]) tensor([0, 7, 5, 6, 4, 8, 1, 3, 4, 1])
